# 07 — RL agent (walk 1, PPO + cost-variant sweep)

PPO policy that allocates weights over the walk-1 ranker top-30 each Friday.
Trains separate policies at 4 transaction-cost levels (5/2/10/20 bps) so one
overnight run produces all the cost-sensitivity ablations for the paper.

**Spec:** `docs/superpowers/specs/2026-05-17-rl-agent-design.md`.

SB3 conventions: `check_env` validation → `EvalCallback` (best by val) →
`VecNormalize` (saved for backtest reuse) → TensorBoard. Single-walk MVP;
walks 2-16 deferred. Backtest = notebook 08.

## A. Setup

In [ ]:
from __future__ import annotations
import json, time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, ProgressBarCallback,
)

from src.utils.io import processed_dir, repo_root
from src.utils.ranker import friday_only, project_text_to_pca, load_walk_pca
from src.utils.rl_env import (
    PortfolioEnv,
    build_scoreboard_from_scored_panel,
)

WALK_ID = 1
TRAIN_START, TRAIN_END = '2002-01-01', '2007-12-31'
VAL_START,   VAL_END   = '2008-01-01', '2008-12-31'

PANEL_DIR  = processed_dir() / 'training_panel'
EMBED_DIR  = processed_dir() / 'finbert_stockday_embed'
RANKER_DIR = repo_root() / 'artifacts' / 'ranker' / f'walk-{WALK_ID:03d}'
OUT_ROOT   = repo_root() / 'artifacts' / 'rl' / f'walk-{WALK_ID:03d}'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Cost sweep — primary first; resumable across costs.
COSTS_BPS = [5, 2, 10, 20]
TOP_K = 30
EPISODE_LENGTH = 52
MAX_WEIGHT = 0.10
TOTAL_TIMESTEPS = 2_000_000
EVAL_FREQ = 10_000
CHECKPOINT_FREQ = 200_000
N_ENVS = 4
RANDOM_STATE = 42

print(f'walk {WALK_ID}; train {TRAIN_START}..{TRAIN_END}, val {VAL_START}..{VAL_END}')
print(f'cost sweep: {COSTS_BPS} bps; {TOTAL_TIMESTEPS:,} timesteps per variant')
print(f'out_root: {OUT_ROOT.relative_to(repo_root())}')

## B. Build the scoreboard

For each Friday in `[train_start, val_end]`, run the walk-1 ranker on every
stock in the universe, keep the top-30 by score. Persist to a parquet so
re-runs / kernel restarts don't re-score the panel.

In [ ]:
SCOREBOARD_PATH = OUT_ROOT / 'scoreboard.parquet'

if SCOREBOARD_PATH.exists():
    scoreboard = pd.read_parquet(SCOREBOARD_PATH)
    print(f'loaded scoreboard from disk: {len(scoreboard)} rows, '
          f'{scoreboard["date"].nunique()} Fridays')
else:
    ranker_bundle = joblib.load(RANKER_DIR / 'model.joblib')
    ranker_model = ranker_bundle['model']
    ranker_features = ranker_bundle['feature_names']
    pca, n_pca = load_walk_pca(WALK_ID)
    print(f'ranker loaded: {len(ranker_features)} features; PCA n_components={n_pca}')

    def _load_panel(start: str, end: str) -> pd.DataFrame:
        s, e = pd.Timestamp(start), pd.Timestamp(end)
        frames = []
        for y in range(s.year, e.year + 1):
            for p in sorted((PANEL_DIR / f'year={y}').glob('*.parquet')):
                df = pd.read_parquet(p)
                df['date'] = pd.to_datetime(df['date'])
                df = df[(df['date'] >= s) & (df['date'] <= e)]
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def _load_embed(start: str, end: str) -> pd.DataFrame:
        s, e = pd.Timestamp(start), pd.Timestamp(end)
        frames = []
        for y in range(s.year, e.year + 1):
            for p in sorted((EMBED_DIR / f'year={y}').glob('*.parquet')):
                df = pd.read_parquet(p, columns=['permno', 'date', 'vec'])
                df['date'] = pd.to_datetime(df['date'])
                df = df[(df['date'] >= s) & (df['date'] <= e)]
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    panel = _load_panel(TRAIN_START, VAL_END)
    embed = _load_embed(TRAIN_START, VAL_END)
    embed_pca = project_text_to_pca(embed, pca)

    # Friday-only + join PCA + drop rows w/o a forward return.
    fri = friday_only(panel).merge(embed_pca, on=['permno', 'date'], how='inner')
    fri = fri.dropna(subset=['fwd_ret_5d']).copy()

    # Build the feature matrix in the same column order the ranker expects.
    # Missing cols (shouldn't happen, but defensive) get NaN.
    X = pd.DataFrame({c: fri[c] if c in fri.columns else np.nan for c in ranker_features})
    fri['score'] = ranker_model.predict(X)

    scoreboard = build_scoreboard_from_scored_panel(fri, top_k=TOP_K)
    scoreboard.to_parquet(SCOREBOARD_PATH, compression='zstd', index=False)
    print(f'wrote scoreboard: {len(scoreboard)} rows, '
          f'{scoreboard["date"].nunique()} Fridays -> '
          f'{SCOREBOARD_PATH.relative_to(repo_root())}')

# Split into train / val views once.
scoreboard['date'] = pd.to_datetime(scoreboard['date'])
scoreboard_train = scoreboard[(scoreboard['date'] >= TRAIN_START) & (scoreboard['date'] <= TRAIN_END)].copy()
scoreboard_val   = scoreboard[(scoreboard['date'] >= VAL_START)   & (scoreboard['date'] <= VAL_END)].copy()
print(f'train Fridays: {scoreboard_train["date"].nunique()}; '
      f'val Fridays: {scoreboard_val["date"].nunique()}')

## C. `train_one(cost_bps, out_dir)` — single cost-variant training

Builds train (4-env DummyVecEnv, randomized starts) and val (1-env, deterministic)
envs at the given cost level, runs PPO. `EvalCallback` saves the best policy
by val mean reward; `CheckpointCallback` writes rolling backups; TensorBoard
logs reward + value loss + entropy.

`VecNormalize` stats are saved alongside the policy so notebook 08's backtest
applies the same observation normalization to 2009 data.

In [ ]:
def _make_env_fn(scoreboard_subset: pd.DataFrame, cost_bps: float, seed: int):
    def _thunk():
        env = PortfolioEnv(
            scoreboard=scoreboard_subset,
            top_k=TOP_K,
            episode_length=EPISODE_LENGTH,
            cost_bps=cost_bps,
            max_weight=MAX_WEIGHT,
        )
        env.reset(seed=seed)
        return env
    return _thunk


def train_one(cost_bps: float, out_dir: Path) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / 'ckpts').mkdir(exist_ok=True)
    (out_dir / 'tb').mkdir(exist_ok=True)

    train_vec = DummyVecEnv([
        _make_env_fn(scoreboard_train, cost_bps, RANDOM_STATE + i)
        for i in range(N_ENVS)
    ])
    train_vec = VecNormalize(train_vec, norm_obs=True, norm_reward=False, clip_obs=10.0)

    val_vec = DummyVecEnv([_make_env_fn(scoreboard_val, cost_bps, RANDOM_STATE + 1000)])
    val_vec = VecNormalize(val_vec, norm_obs=True, norm_reward=False, clip_obs=10.0,
                           training=False)

    eval_cb = EvalCallback(
        val_vec,
        best_model_save_path=str(out_dir),
        log_path=str(out_dir),
        eval_freq=max(EVAL_FREQ // N_ENVS, 1),
        n_eval_episodes=1,
        deterministic=True,
    )
    ckpt_cb = CheckpointCallback(
        save_freq=max(CHECKPOINT_FREQ // N_ENVS, 1),
        save_path=str(out_dir / 'ckpts'),
        name_prefix='ppo',
    )

    model = PPO(
        'MlpPolicy',
        train_vec,
        policy_kwargs=dict(net_arch=[256, 128]),
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.0,
        vf_coef=0.5,
        max_grad_norm=0.5,
        device='cpu',
        tensorboard_log=str(out_dir / 'tb'),
        seed=RANDOM_STATE,
        verbose=0,
    )

    t0 = time.time()
    model.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        callback=[eval_cb, ckpt_cb, ProgressBarCallback()],
        progress_bar=False,  # ProgressBarCallback handles it
    )
    elapsed = time.time() - t0

    model.save(out_dir / 'final_policy')
    train_vec.save(str(out_dir / 'vec_normalize.pkl'))

    metrics = {
        'cost_bps': cost_bps,
        'total_timesteps': TOTAL_TIMESTEPS,
        'n_envs': N_ENVS,
        'wall_time_sec': elapsed,
        'wall_time_hr': elapsed / 3600.0,
        'best_val_mean_reward': float(eval_cb.best_mean_reward),
    }
    (out_dir / 'training_metrics.json').write_text(json.dumps(metrics, indent=2))
    print(f'  cost={cost_bps}bps done; wall={elapsed/60:.1f} min; '
          f'best_val_mean_reward={metrics["best_val_mean_reward"]:.4f}')
    return metrics

## D. Train all cost variants

Resumable: any cost variant whose `final_policy.zip` already exists is skipped.
Long-running — designed to be kicked off and left overnight.

In [ ]:
all_metrics = []
for cost_bps in COSTS_BPS:
    out_dir = OUT_ROOT / f'cost-{cost_bps:03d}bps'
    final_path = out_dir / 'final_policy.zip'
    if final_path.exists():
        existing = json.loads((out_dir / 'training_metrics.json').read_text())
        print(f'cost={cost_bps}bps: exists '
              f'(best_val_mean_reward={existing.get("best_val_mean_reward", float("nan")):.4f}) — skipping')
        all_metrics.append(existing)
        continue
    print(f'\n=== training cost={cost_bps}bps -> {out_dir.relative_to(repo_root())} ===')
    m = train_one(float(cost_bps), out_dir)
    all_metrics.append(m)

(OUT_ROOT / 'all_costs_summary.json').write_text(
    json.dumps({'cost_variants': all_metrics}, indent=2)
)
print(f'\nfinished {len(all_metrics)} cost variants; '
      f'summary -> {(OUT_ROOT / "all_costs_summary.json").relative_to(repo_root())}')

## E. Cross-cost diagnostics

Per-cost summary table + bar plot of best val mean reward. Intuitive read on
whether the policy's quality degrades as transaction costs rise (the expected
direction). True comparison vs baselines happens in notebook 08.

In [ ]:
import matplotlib.pyplot as plt

summary = pd.DataFrame(all_metrics).sort_values('cost_bps').reset_index(drop=True)
display_cols = ['cost_bps', 'total_timesteps', 'wall_time_hr', 'best_val_mean_reward']
print('per-cost-variant summary:')
print(summary[display_cols].to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(summary['cost_bps'].astype(str) + ' bps',
       summary['best_val_mean_reward'], color='steelblue')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('transaction cost')
ax.set_ylabel('best val mean reward (1-year episode)')
ax.set_title('PPO val reward across cost regimes (walk 1)')
plt.tight_layout()
plt.show()